# Attention and language-model experiments
Coursework-derived implementations with causal-mask corrections. Synthetic training curves are optimization demonstrations, not language benchmarks. Optional AG News tokenization analysis downloads data.


### Experiment cell 1


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

def scaled_dot_product_attention(query, key, value, causal_mask=False):

    d_k = query.shape[-1]
    seq_len = query.shape[-2]


    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)


    if causal_mask:
        # mask[i,j] = True  means position i is NOT allowed to attend to j
        mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=query.device), diagonal=1)
        scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))

    # softmax over the key dimension
    attention_weights = F.softmax(scores, dim=-1)

    # weighted sum of values
    output = torch.matmul(attention_weights, value)

    return output, attention_weights


# Test without mask
torch.manual_seed(42)
query = torch.rand(1, 3, 64)
key   = torch.rand(1, 3, 64)
value = torch.rand(1, 3, 64)

out_no_mask, attn_no_mask = scaled_dot_product_attention(query, key, value, causal_mask=False)
print(f"Output shape (no mask):      {out_no_mask.shape}")
print(f"Attention weights (no mask):\n{attn_no_mask[0].detach().numpy().round(3)}")

# Test with causal mask
out_mask, attn_mask = scaled_dot_product_attention(query, key, value, causal_mask=True)
print(f"\nOutput shape (causal mask):      {out_mask.shape}")
print(f"Attention weights (causal mask):\n{attn_mask[0].detach().numpy().round(3)}")

# Visualise attention matrices
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, weights, title in zip(
    axes,
    [attn_no_mask[0].detach(), attn_mask[0].detach()],
    ["Full (bidirectional) Attention", "Causal (autoregressive) Attention"]
):
    im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('attention_weights.png', dpi=120)
plt.show()
print("\nNote: upper-triangle in causal attention is zero — future positions are masked.")

### Experiment cell 2


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def positional_encoding(seq_len, d_model):

    position = np.arange(seq_len)[:, np.newaxis]          # (seq_len, 1)
    div_term = np.exp(
        np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model)
    )                                                       # (d_model/2,)

    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(position * div_term)  # even dimensions
    pe[:, 1::2] = np.cos(position * div_term)  # odd  dimensions
    return pe


# Compute and display
pos_enc = positional_encoding(50, 64)
print(f"Positional encoding shape: {pos_enc.shape}")

### Experiment cell 3


In [ ]:
import torch
import torch.nn as nn

class TransformerDecoderBlock(nn.Module):


    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn   = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads,
                                                  dropout=dropout, batch_first=False)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),          # dropout inside FFN
            nn.Linear(d_ff, d_model)
        )
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)   # after attention
        self.dropout2 = nn.Dropout(dropout)   # after FFN

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        seq_len = x.size(0)

        # Build causal mask: upper triangular (seq_len × seq_len)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device), diagonal=1
        ).bool()


        attn_output, _ = self.self_attn(x, x, x, attn_mask=causal_mask)
        x = self.norm1(x + self.dropout1(attn_output))

        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_output))

        return x



torch.manual_seed(0)
decoder_block = TransformerDecoderBlock(d_model=64, num_heads=8, d_ff=256, dropout=0.1)
decoder_block.eval()   # disable dropout for deterministic output

x   = torch.rand(10, 32, 64)
out = decoder_block(x)
print(f"Input  shape: {x.shape}")
print(f"Output shape: {out.shape}")
print(f"\nParameter count: {sum(p.numel() for p in decoder_block.parameters()):,}")
print("Output mean / std:", out.mean().item(), "/", out.std().item())

### Experiment cell 4


In [ ]:
import torch
import torch.nn as nn

class DecoderOnlyTransformer(nn.Module):


    def __init__(self, d_model: int, num_heads: int, d_ff: int,
                 num_layers: int, vocab_size: int, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=num_heads,
                dim_feedforward=d_ff, dropout=dropout,
                batch_first=False
            )
            for _ in range(num_layers)
        ])
        self.ln         = nn.LayerNorm(d_model)
        # Final linear projection: maps d_model → vocab_size
        self.lm_head    = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (seq_len, batch_size, d_model)  — pre-embedded tokens
        Returns:
            logits: (seq_len, batch_size, vocab_size)
        """
        seq_len = x.size(0)
        # Causal mask prevents attending to future positions
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)

        for layer in self.layers:
            # Causal self-attention without an unmasked cross-attention path.
            x = layer(x, src_mask=causal_mask)

        x = self.ln(x)
        logits = self.lm_head(x)   # project to vocabulary
        return logits



VOCAB_SIZE = 10_000
torch.manual_seed(0)

model = DecoderOnlyTransformer(
    d_model=64, num_heads=8, d_ff=256,
    num_layers=3, vocab_size=VOCAB_SIZE
)

x      = torch.rand(10, 32, 64)    # (seq_len=10, batch=32, d_model=64)
logits = model(x)

print(f"Input  shape:  {x.shape}")
print(f"Output (logits) shape: {logits.shape}")
print(f"Expected:              (10, 32, {VOCAB_SIZE})")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

### Experiment cell 5


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


class SimpleLLM(nn.Module):


    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.ln    = nn.LayerNorm(d_model)
        self.fc    = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        return self.fc(self.ln(self.embed(x)))


def train_model(d_model: int, vocab_size: int = 10_000,
                num_iters: int = 100, lr: float = 1e-3,
                seed: int = 42) -> list:
    """Train a SimpleLLM and return per-iteration loss history."""
    torch.manual_seed(seed)
    model     = SimpleLLM(d_model, vocab_size)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn   = nn.CrossEntropyLoss()

    dummy_input  = torch.randint(0, vocab_size, (32, 20))
    dummy_target = torch.randint(0, vocab_size, (32, 20))

    history = []
    for _ in range(num_iters):
        optimizer.zero_grad()
        output = model(dummy_input)                          # (32, 20, vocab)
        loss   = loss_fn(output.view(-1, vocab_size), dummy_target.view(-1))
        loss.backward()
        optimizer.step()
        history.append(loss.item())

    return history


# Train models of different sizes
model_sizes  = [128, 256, 512]
num_iters    = 150
loss_curves  = {}

for d_model in model_sizes:
    losses = train_model(d_model=d_model, num_iters=num_iters)
    loss_curves[d_model] = losses
    num_params = d_model * 10_000 + d_model + d_model * 10_000  # approx
    print(f"d_model={d_model:>3d}  Final loss: {losses[-1]:.4f} "
          f" Min loss: {min(losses):.4f}")

# Plot loss curves
plt.figure(figsize=(10, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']
for (d_model, losses), color in zip(loss_curves.items(), colors):
    plt.plot(losses, label=f'd_model={d_model}', color=color, linewidth=2)

plt.xlabel('Training iteration', fontsize=13)
plt.ylabel('Cross-entropy loss', fontsize=13)
plt.title('Scaling Effect: Loss vs. Iterations for Different Model Sizes', fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('scaling_loss.png', dpi=120)
plt.show()

print("\nObservation: Larger d_model generally converges to a lower final loss,"
      " illustrating neural scaling laws.")

### Experiment cell 6


In [ ]:
# Install required packages (run once in Colab)

### Experiment cell 7


In [ ]:
import collections
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datasets import load_dataset
from transformers import AutoTokenizer



dataset = load_dataset("ag_news", split="train")


text_corpus = " ".join(dataset[:500]["text"])
print(f"Corpus size: {len(text_corpus):,} characters")



def whitespace_tokenize(text: str) -> list:
    """Simplest: split on whitespace."""
    return text.lower().split()


def character_tokenize(text: str) -> list:
    """Character-level tokenization; exclude pure whitespace tokens."""
    return [ch for ch in text if ch.strip()]


def subword_tokenize(text: str, tokenizer) -> list:
    """BPE/WordPiece subword tokenization via HuggingFace tokenizer."""
    return tokenizer.tokenize(text)




def analyze_token_distribution(text_corpus: str, top_n: int = 50):

    bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    methods = {
        "Whitespace": whitespace_tokenize(text_corpus),
        "Subword (BERT BPE)": subword_tokenize(text_corpus, bert_tokenizer),
        "Character": character_tokenize(text_corpus),
    }

    fig, axes = plt.subplots(3, 1, figsize=(16, 14))
    colors = ['#3498db', '#e67e22', '#2ecc71']

    stats = {}
    for ax, (method_name, tokens), color in zip(axes, methods.items(), colors):
        counter       = collections.Counter(tokens)
        most_common   = counter.most_common(top_n)
        tok_labels, counts = zip(*most_common)

        ax.bar(range(len(tok_labels)), counts, color=color, alpha=0.85)
        ax.set_xticks(range(len(tok_labels)))
        ax.set_xticklabels(tok_labels, rotation=75, ha='right', fontsize=7)
        ax.set_ylabel('Frequency', fontsize=11)
        ax.set_title(f'{method_name} Tokenization — Top {top_n} Tokens', fontsize=12)
        ax.grid(axis='y', alpha=0.3)

        stats[method_name] = {
            'vocab_size': len(counter),
            'total_tokens': len(tokens),
            'top_token': most_common[0],
        }

    plt.tight_layout()
    plt.savefig('token_distributions.png', dpi=120)
    plt.show()

    # Print summary statistics
    print("\nTokenization Statistics ")
    for method, s in stats.items():
        print(f"{method}:")
        print(f"  Vocabulary size : {s['vocab_size']:,}")
        print(f"  Total tokens    : {s['total_tokens']:,}")
        print(f"  Top token       : '{s['top_token'][0]}'  ({s['top_token'][1]:,} times)")

    return stats


# Run analysis
stats = analyze_token_distribution(text_corpus, top_n=50)